# Sequencing analysis (short version)

### Functions used for analysis

In [9]:
import heapq
from collections import Counter
import numpy as np  

def find_overlap(seq1, seq2):
    """Find the longest overlap between the end of seq1 and the start of seq2."""
    overlap_length = 0
    for i in range(1, min(len(seq1), len(seq2)) + 1):
        if seq1[-i:] == seq2[:i]:
            overlap_length = i
    return overlap_length

def combine_sequences(seq1, seq2):
    """Combine two sequences by removing the overlapping part from the second sequence."""
    overlap_length = find_overlap(seq1, seq2)
    return seq1 + seq2[overlap_length:]
def organize_reads(input_file, output_file):
    with open(input_file, 'r') as file:
        sequences = [line.strip() for line in file.readlines()]
    for i in range(len(sequences)):
        if ' ' not in sequences[i]:
            continue
        seq1,seq2=sequences[i].split(' ')
        combined_sequence = combine_sequences(seq1, seq2)
        sequences[i]=combined_sequence
    with open(output_file, 'w') as file:
        for sequence in sequences:
            file.write(sequence + '\n')
def extract_sequences(input_file, output_file):
    target_start = 'GCGATCGC'
    target_end = 'GAATTCTGCAGTCGACGGTAC'
    
    with open(input_file, 'r') as infile, open(output_file, 'w') as outfile:
        for line in infile:
            if target_start in line and target_end in line:
                parts = line.split(target_start)
                if len(parts) > 2:
                    # Extract between second occurrence of start and first occurrence of end
                    sub_part = target_start.join(parts[2:])  # Get the part after the second GCGATCGC
                    seq = sub_part.split(target_end)[0]  # Get up to first GAATTCT...
                else:
                    # If there's only one occurrence of GCGATCGC
                    sub_part = parts[1]  # Get the part after the first GCGATCGC
                    seq = sub_part.split(target_end)[0]  # Get up to first GAATTCT...
                outfile.write(seq + '\n')
            elif target_start in line:
                # Only the start is present
                sub_part = line.split(target_start)[1]
                seq = sub_part.split(target_end)[0]
                outfile.write(seq + '\n')
def extract_lines(input_file, output_file):
    lines = []
    with open(input_file, 'r') as file:
        for line in file:
            if len(line.strip()) == 376:
                lines.append(line)
    with open(output_file, 'w') as file:
        for line in lines:
            file.write(line)

In [10]:
def read_fasta(filename):
    """Reads sequences from a FASTA file."""
    sequences = []
    with open(filename, 'r') as file:
        for line in file:
            if not line.startswith('>'):
                sequences.append(line.strip())
    return sequences

def calculate_base_frequencies(sequences):
    """Calculates the frequency of each base at every position."""
    sequence_length = len(sequences[0])
    base_counts = [Counter() for _ in range(sequence_length)]

    for seq in sequences:
        for i, base in enumerate(seq):
            base_counts[i][base] += 1

    base_frequencies = []
    for counts in base_counts:
        total = sum(counts.values())
        frequencies = {base: count / total for base, count in counts.items()}
        base_frequencies.append(frequencies)
    return base_frequencies

def apply_cutoff(base_frequencies):
    """Keeps only the top 2 most frequent bases at each position."""
    filtered = []
    for freq in base_frequencies:
        # Sort bases by frequency (descending), keep top 2
        top2 = dict(sorted(freq.items(), key=lambda x: x[1], reverse=True)[:2])
        filtered.append(top2)
    return filtered

def replace_with_combined_bases(frequencies):
    """Replaces bases with IUPAC combined bases according to the frequencies."""
    iupac_codes = {
        frozenset(['A', 'G']): 'R', frozenset(['C', 'T']): 'Y',
        frozenset(['G', 'T']): 'K', frozenset(['A', 'C']): 'M',
        frozenset(['G', 'C']): 'S', frozenset(['A', 'T']): 'W',
        frozenset(['A']): 'A', frozenset(['C']): 'C',
        frozenset(['G']): 'G', frozenset(['T']): 'T',
    }

    consensus_sequence = ''
    for pos_freq in frequencies:
        significant_bases = frozenset(pos_freq.keys())
        consensus_sequence += iupac_codes.get(significant_bases, 'A')  # 'N' if no significant bases or combination not in table

    return consensus_sequence
def decryption  (consensus_sequence,obfuscation_seqs):
    combined_bases = {'R': ['A', 'G'], 'Y': ['C', 'T'], 'K': ['G', 'T'], 'M': ['A', 'C'], 'S': ['C', 'G'], 'W': ['A', 'T']}
    decrypted=''
    for i in range(len(consensus_sequence)):
        if consensus_sequence[i] == obfuscation_seqs[i]:
            decrypted+=consensus_sequence[i]
        else:
            choice=[b for b in combined_bases[consensus_sequence[i]] if b != obfuscation_seqs[i]]
            decrypted+=choice[0]
    return decrypted

### Processing Illumina Fastq files, extracting payload sequence

In [ ]:
import os
from os import listdir
from os.path import isfile, join
files = [f'/home/zzk/work/DNA_storage/30-1089119466/Step1/{f}' for f in listdir('/home/zzk/work/DNA_storage/30-1089119466/Step1/') if isfile(join('/home/zzk/work/DNA_storage/30-1089119466/Step1/', f)) and f.endswith('.fastq')]
for file in files:
   os.system(f'awk \'NR%4 ==2\' {file} > /home/zzk/work/DNA_storage/30-1089119466//Step2/{file[file.rfind("/")+1:]}')  # Extract the sequences from the fastq files


In [ ]:
files = [f'/home/zzk/work/DNA_storage/30-1089119466/Step2/{f}' for f in listdir('/home/zzk/work/DNA_storage/30-1089119466/Step2/') if isfile(join('/home/zzk/work/DNA_storage/30-1089119466/Step2/', f)) and f.find("R2")!=-1]
for file in files:
    print(file)
    os.system(f'awk \'{{print $0}}\' {file} | rev | tr \'ATCGN\' \'TAGCN\' > /home/zzk/work/DNA_storage/30-1089119466/Step3/{file[file.rfind("/")+1:]}')  # reverse complement the R2 sequences


In [ ]:
files = [f'/home/zzk/work/DNA_storage/30-1089119466/Step2/{f}' for f in listdir('/home/zzk/work/DNA_storage/30-1089119466/Step2/') if isfile(join('/home/zzk/work/DNA_storage/30-1089119466/Step2/', f)) and f.find("R1") != -1]
for file in files:
    file1 = file
    file2 = file.replace("Step2","Step3").replace("R1", "R2")
    print(file1, file2)
    os.system(f'paste -d " " {file1} {file2} > /home/zzk/work/DNA_storage/30-1089119466/Step4/{os.path.basename(file)}')


In [ ]:
files = [f'/home/zzk/work/DNA_storage/30-1089119466/Step4/{f}' for f in listdir('/home/zzk/work/DNA_storage/30-1089119466/Step4/') if isfile(join('/home/zzk/work/DNA_storage/30-1089119466/Step4/', f)) and f.endswith('.fastq')]
for file in files:
    os.system(f'grep -E \".*AGCTGGGACCACCTTATATTCCCAG.*GAATTCTGCAGTCGACGGTACC.*\" {file} > /home/zzk/work/DNA_storage/30-1089119466/Step5/{file[file.rfind("/")+1:]}')  # filter the sequences with the primers


In [ ]:
files=[f'/home/zzk/work/DNA_storage/30-1089119466/Step5/{f}' for f in listdir('/home/zzk/work/DNA_storage/30-1089119466/Step5/') if isfile(join('/home/zzk/work/DNA_storage/30-1089119466/Step5/', f)) and f.endswith('.fastq')]
for file in files:
    output=file.replace("Step5","Step6")
    organize_reads(file,output) # combine the sequences without the overlapping part


In [ ]:
files=[f'/home/zzk/work/DNA_storage/30-1089119466/Step6/{f}' for f in listdir('/home/zzk/work/DNA_storage/30-1089119466/Step6/') if isfile(join('/home/zzk/work/DNA_storage/30-1089119466/Step6/', f)) and f.endswith('.fastq')]
for file in files:
    output=file.replace("Step6","Step7")
    extract_sequences(file,output) # extract the sequences between the primers
files=[f'/home/zzk/work/DNA_storage/30-1089119466/Step7/{f}' for f in listdir('/home/zzk/work/DNA_storage/30-1089119466/Step7/') if isfile(join('/home/zzk/work/DNA_storage/30-1089119466/Step7/', f)) and f.endswith('.fastq')]
for file in files:
    output=file.replace("Step7","Step8")
    extract_lines(file,output) # extract the sequences with the length of 376, the length which was determined by the obfuscation data                    


### calculating per-position base frequencies for each complete fragment, and generating consensus sequence

In [ ]:
import os
import csv
from collections import Counter
files=[f'/home/zzk/work/DNA_storage/30-1089119466/Step8/{f}' for f in listdir('/home/zzk/work/DNA_storage/30-1089119466/Step8/') if isfile(join('/home/zzk/work/DNA_storage/30-1089119466/Step8/', f)) and f.endswith('.fastq')]
output_dir='/home/zzk/work/DNA_storage/30-1089119466/base_frequnceies/'
ref="KMMKMSKYRRSMWYSMRSWSKKMYMWWKRYSSRYRSYMRKRKRRRMKWKWRWWMWKSKRMKMRMSKSMMRRKSRYRRKMMYMKRKSRMYWYYWYYRMYYYSYRSWYKWWRSMRKMWRMYYMRSKMSWRSMRKYRRMRYWWRSSRKKYSSRMYKKWKKKKRWKKRYRWRWMMWWWSMMMWWYRKMKYMSMRWMSWSYSKMSMWYKMMKRRYSKWYMMKSRKKWWSWSSWWYYSMWMYWYSYMSMYKWRWRMRKMWRRYYMRRKWMWMSMRRYRSMRYWRSSRRYRKRRMKWKRWYRKYRKMRSWRSYRSYRKMRMYWKMWMRKYRKWRYWKMMKYMWYMWWRRYKWWWKMRKMKSWYYMWRKRMWWSMRSYRRRRMW"
for file in files:
    sequences = read_fasta(file)
    base_frequencies = calculate_base_frequencies(sequences)
    # save the base frequencies the first 10 bases into a csv file and each base is one column
    csv_path = os.path.join(output_dir, os.path.basename(file).replace('.fastq', '.csv'))
    with open(csv_path, mode='w') as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(['Base'] + list(range(1, 377)))
        for base in ['A', 'C', 'G', 'T']:
            frequencies = [freq.get(base, 0) for freq in base_frequencies[:376]]
            writer.writerow([base] + frequencies)
    filtered_frequencies = apply_cutoff(base_frequencies)
    consensus_sequence = replace_with_combined_bases(filtered_frequencies)
    print(os.path.basename(file), consensus_sequence == ref)

# Sequencing analysis (for extended  payload, incorporating index)

In [1]:
from __future__ import annotations

import csv
import re
from collections import Counter, defaultdict
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterator, List, Tuple

In [ ]:
REVCOMP_TRANS = str.maketrans("ACGTN", "TGCAN")
PATTERN = re.compile(r"AAACAAA([ACGTN])TCGAAGTCA([A-Z]*)TCTCACTCGGTTT") # The pattern to filter the sequences.

In [3]:
@dataclass
class ReadRecord:
    read_id: str
    sequence: str

### Functions for processing the sequencing data

In [4]:
def normalize_read_id(header: str) -> str:
    read_id = header.strip()
    if read_id.startswith("@"):
        read_id = read_id[1:]
    read_id = read_id.split()[0]
    if read_id.endswith("/1") or read_id.endswith("/2"):
        read_id = read_id[:-2]
    return read_id


def read_fastq_sequences(path: Path) -> Iterator[ReadRecord]:
    with path.open() as handle:
        while True:
            header = handle.readline().strip()
            if not header:
                break
            sequence = handle.readline().strip()
            handle.readline()
            handle.readline()
            if not sequence:
                continue
            yield ReadRecord(normalize_read_id(header), sequence.upper())


def reverse_complement(seq: str) -> str:
    return seq.translate(REVCOMP_TRANS)[::-1]


def find_max_overlap(r1: str, r2: str) -> int:
    max_len = min(len(r1), len(r2))
    for overlap in range(max_len, 0, -1):
        if r1[-overlap:] == r2[:overlap]:
            return overlap
    return 0


def merge_reads(r1: str, r2: str) -> Tuple[str, int]:
    overlap = find_max_overlap(r1, r2)
    if overlap == 0:
        return "", 0
    merged = r1 + r2[overlap:]
    return merged, overlap

In [5]:
def collect_fastq_pairs(directory: Path) -> Dict[str, Tuple[Path, Path]]:
    pairs: Dict[str, Tuple[Path, Path]] = {}
    for r1_path in directory.glob("*_R1_*.fastq"):
        r2_path = Path(str(r1_path).replace("_R1_", "_R2_"))
        if not r2_path.exists():
            continue
        key = r1_path.name.split("_R1_")[0]
        pairs[key] = (r1_path, r2_path)
    return pairs


def compute_position_frequencies(sequences: List[str]) -> List[Dict[str, int]]:
    if not sequences:
        return []
    length = len(sequences[0])
    frequencies: List[Dict[str, int]] = []
    for idx in range(length):
        counter = Counter(seq[idx] for seq in sequences)
        row = {
            "Position": idx + 1,
            "A": counter.get("A", 0),
            "C": counter.get("C", 0),
            "G": counter.get("G", 0),
            "T": counter.get("T", 0),
            "N": counter.get("N", 0),
        }
        known_total = sum(row[base] for base in ("A", "C", "G", "T", "N"))
        row["Other"] = sum(counter.values()) - known_total
        row["Total"] = sum(counter.values())
        frequencies.append(row)
    return frequencies


def write_frequency_csv(path: Path, frequencies: List[Dict[str, int]]) -> None:
    if not frequencies:
        return
    with path.open("w", newline="") as handle:
        fieldnames = ["Position", "A", "C", "G", "T", "N", "Other", "Total"]
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        for row in frequencies:
            writer.writerow(row)


def write_fasta(path: Path, records: List[Tuple[str, str]]) -> None:
    if not records:
        return
    with path.open("w") as handle:
        for read_id, sequence in records:
            handle.write(f">{read_id}\n{sequence}\n")


def select_dominant_length(records: List[Tuple[str, str]]) -> Tuple[int, List[Tuple[str, str]]]:
    if not records:
        return 0, []
    length_counts = Counter(len(seq) for _, seq in records)
    dominant_length, _ = max(length_counts.items(), key=lambda item: (item[1], item[0]))
    filtered = [(rid, seq) for rid, seq in records if len(seq) == dominant_length]
    return dominant_length, filtered

In [ ]:
def process_pair(r1_path: Path, r2_path: Path, output_root: Path) -> Tuple[List[Dict[str, str]], List[Dict[str, str]]]:
    r1_sequences: Dict[str, str] = {}
    for record in read_fastq_sequences(r1_path):
        r1_sequences[record.read_id] = record.sequence

    r2_sequences: Dict[str, str] = {}
    for record in read_fastq_sequences(r2_path):
        r2_sequences[record.read_id] = record.sequence

    r2_revcomp = {rid: reverse_complement(seq) for rid, seq in r2_sequences.items()}

    paired_ids = sorted(set(r1_sequences) & set(r2_revcomp))

    merged_reads: Dict[str, str] = {}
    for read_id in paired_ids:
        merged, overlap = merge_reads(r1_sequences[read_id], r2_revcomp[read_id])
        if overlap > 0 and merged:
            merged_reads[read_id] = merged

    grouped_sequences: Dict[str, List[Tuple[str, str]]] = defaultdict(list)
    pattern_matched_count = 0

    dataset_name = r1_path.name.split("_R1_")[0]
    dataset_output = output_root / dataset_name
    dataset_output.mkdir(parents=True, exist_ok=True)
# classfication and grouping by Index for F1/F2/F3:A/G/C
    for read_id, sequence in merged_reads.items():
        match = PATTERN.search(sequence)
        if not match:
            continue
        pattern_matched_count += 1
        index = "AAA" + match.group(1)
        payload = match.group(2).upper()
        grouped_sequences[index].append((read_id, payload))

    index_summaries: List[Dict[str, str]] = []
    counts_summary: List[Dict[str, str]] = []
    final_grouped: Dict[str, List[Tuple[str, str]]] = {}

    for index, records in grouped_sequences.items():
        dominant_length, dominant_records = select_dominant_length(records)
        final_grouped[index] = dominant_records
        index_summaries.append(
            {
                "dataset": dataset_name,
                "index": index,
                "pattern_matched_reads": str(len(records)),
                "dominant_length": str(dominant_length),
                "final_reads": str(len(dominant_records)),
            }
        )
        if dominant_records:
            fasta_path = dataset_output / f"{dataset_name}_{index}_reads.fasta"
            write_fasta(fasta_path, dominant_records)
            frequency_path = dataset_output / f"{dataset_name}_{index}_frequencies.csv"
            write_frequency_csv(
                frequency_path,
                compute_position_frequencies([seq for _, seq in dominant_records])
            )

    counts_summary.extend(
        [
            {"dataset": dataset_name, "step": "Step1_R1_sequences", "count": str(len(r1_sequences))},
            {"dataset": dataset_name, "step": "Step1_R2_sequences", "count": str(len(r2_sequences))},
            {"dataset": dataset_name, "step": "Step3_paired_reads", "count": str(len(paired_ids))},
            {"dataset": dataset_name, "step": "Step3_overlap_merged", "count": str(len(merged_reads))},
            {"dataset": dataset_name, "step": "Step5_pattern_matched", "count": str(pattern_matched_count)},
            {
                "dataset": dataset_name,
                "step": "Step6_index_grouped",
                "count": str(sum(len(records) for records in grouped_sequences.values())),
            },
            {
                "dataset": dataset_name,
                "step": "Step7_dominant_length",
                "count": str(sum(len(records) for records in final_grouped.values())),
            },
        ]
    )

    return counts_summary, index_summaries

In [7]:
def run_pipeline(base_dir: Path) -> Tuple[Path, Path]:
    output_root = base_dir / "pipeline_output"
    output_root.mkdir(parents=True, exist_ok=True)

    pairs = collect_fastq_pairs(base_dir)
    if not pairs:
        raise RuntimeError("No FASTQ pairs found.")

    all_counts: List[Dict[str, str]] = []
    all_index_summaries: List[Dict[str, str]] = []

    for dataset, (r1_path, r2_path) in pairs.items():
        counts, index_summary = process_pair(r1_path, r2_path, output_root)
        all_counts.extend(counts)
        all_index_summaries.extend(index_summary)

    counts_path = output_root / "pipeline_counts.csv"
    with counts_path.open("w", newline="") as handle:
        fieldnames = ["dataset", "step", "count"]
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(all_counts)

    index_summary_path = output_root / "index_summary.csv"
    with index_summary_path.open("w", newline="") as handle:
        fieldnames = ["dataset", "index", "pattern_matched_reads", "dominant_length", "final_reads"]
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(all_index_summaries)

    return counts_path, index_summary_path

### Run the analysis

In [8]:
# Adjust the base directory if this notebook is moved.
base_dir = Path("/home/zzk/work/DNA_storage/30-1270800528/00_fastq/Step1")
counts_csv, summary_csv = run_pipeline(base_dir)
counts_csv, summary_csv

(PosixPath('/home/zzk/work/DNA_storage/30-1270800528/00_fastq/Step1/pipeline_output/pipeline_counts.csv'),
 PosixPath('/home/zzk/work/DNA_storage/30-1270800528/00_fastq/Step1/pipeline_output/index_summary.csv'))

### generating consensus sequence

In [ ]:
import csv
from pathlib import Path
from typing import Dict, List
def load_consensus(frequency_csv: Path) -> str:
    rows = []
    with frequency_csv.open() as h:
        header = h.readline().strip().split(",")
        idx_map = {name: i for i, name in enumerate(header)}
        for line in h:
            parts = line.strip().split(",")
            if len(parts) < len(header):
                continue
            rows.append(parts)

    bases = ["A", "C", "G", "T"]  # tie-break order
    pair_to_iupac = {
        frozenset({"A", "C"}): "M",
        frozenset({"A", "G"}): "R",
        frozenset({"A", "T"}): "W",
        frozenset({"C", "G"}): "S",
        frozenset({"C", "T"}): "Y",
        frozenset({"G", "T"}): "K",
    }

    consensus = []
    for r in rows:
        counts = {b: int(r[idx_map[b]]) if b in idx_map else 0 for b in bases}
        # sort by count desc, then by predefined order for deterministic ties
        sorted_bases = sorted(bases, key=lambda b: (-counts[b], bases.index(b)))

        top = sorted_bases[0]
        if counts[top] == 0:
            n_count = int(r[idx_map["N"]]) if "N" in idx_map else 0
            consensus.append("N" if n_count > 0 else "N")
            continue

        # pick the second most frequent non-zero base (distinct)
        second = None
        for b in sorted_bases[1:]:
            if counts[b] > 0:
                second = b
                break

        if second is None:
            consensus.append(top)
        else:
            consensus.append(pair_to_iupac[frozenset({top, second})])

    return "".join(consensus)


def build_combined_consensus(output_root: Path, indices: List[str]) -> List[Dict[str, str]]:
    results = []
    for dataset_dir in output_root.iterdir():
        if not dataset_dir.is_dir():
            continue
        dataset = dataset_dir.name
        consensus_parts = []
        per_index = {}
        for idx in indices:
            freq_path = dataset_dir / f"{dataset}_{idx}_frequencies.csv"
            if freq_path.exists():
                seq = load_consensus(freq_path)
                per_index[idx] = seq
                consensus_parts.append(seq)
            else:
                per_index[idx] = ""
        combined = "".join(consensus_parts)
        row = {"dataset": dataset, "combined_consensus": combined}
        for idx in indices:
            row[f"consensus_{idx}"] = per_index[idx]
        results.append(row)
    return results

base_dir = Path("/home/zzk/work/DNA_storage/30-1270800528/00_fastq/Step1")
output_root = base_dir / "pipeline_output"
indices_to_concat = ["AAAA", "AAAC", "AAAG"]
combined = build_combined_consensus(output_root, indices_to_concat)

# Write combined consensus CSV
combined_path = output_root / "combined_consensus.csv"
with combined_path.open("w", newline="") as h:
    fieldnames = ["dataset"] + [f"consensus_{i}" for i in indices_to_concat] + ["combined_consensus"]
    writer = csv.DictWriter(h, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(combined)

combined_path, combined

In [ ]:
ref_seq="MRRRKWKWYKMYKYSWWSRKRMRWWYWSKKWSYYWMSWWMKYRWRKRWYKSRKWMRMSRMKMRYWSKRYKWYRWMMWWWYYKKRRMYSYSRWYYRYMRSYKMRMMRRWYKMRKRYRYWYKRSSYKWKKRWYKWRSYYKYWKSYKMYKWMWKWWYWKMRSMKMSKRKWWKMRKYYWMSYYSRYKWRMYRRWSSYSKSRRWSMRWRMMSWMRRKMYSMMSKWYSRYYYMKRRRMKKWRYMWSKMWMWKSKKKWYWMMWKRYMKRMMRWYWRMWMRMRMKWKYRKWSRYKRMKMRMRWRSWYYWMRRSYKMKYKWYMWRSMYYRMRWKSYYRMRMWSWKYRKSYSSMWWKWKKRYRKRSSWYKRWKYSMYKKMMMMRRSSYRKYRSRWMSWWRSWYSYSMYKYMSWSMMKSRWSWWKYMKSWMMKSRMRWWSKRSWSMRYMKKWKWSRSWYRKMMYWSSRMMKWWMRYMMYRMRKRWYRYKWRMKRMWKMMYKWWSWSWWSWWWMSKSMKSWRWRSYWWMYYRYRSWSSSMRRRMWSKWKRWYSWMMKRMYKMRYKRMSRMRKWRKRMYSMKMMWRMMSYYWWKRRKWSYRWKYSMRSRWSRRYMRRYRRSYWRWWSSWYSMMKMYKYYRSYWKSYSMRSMWRKSWMRKWRMKYMKMWWSMMRSMRMRSKYRYKMRSWWRWYMRMSWYKRWMWWYWRSWSYMSWYMMSMRSMYSYKMRSWKSWSRSSMYRWYKWRKMWMSYWKWMRYMMMRKWRRSRRRMMSWWWYKKWMRKMSWWYSMSWRWMKYKMMRMKWKMYKRMYRYSWKMWRMSWWSMMRSMWKRWMRKKKSSSRRMMRRWKMYKRYWWRWSKYKRMWMMYRRWKKMMRKRRWKMSSSMRYKSWSYRWRMWSKRMYRWMMMSSMMWYWSSRSMYMWYMMMWSYMSYSKRSMWYRSWYKWYKYWMKSMSRKYRKRSSYRKKMMRYYKRRSWYRKRMRMWMWSSYKMRRWMYRWKYSWRMMRKWMYSRYMRRKWYYWRMSYKWYSYMSWYWWRRSRWMKWSYRKMYWSYRRMYYRSMRYYMRYWWRSRMKYSYRMRMYMWSWSWWWSMSMWRYMRRWWYKKRMRMRRRSSYKMRRRRSRSYMYSWMYRYKSYYMYMWKWMYYRRKYMKWWMRRMWKRRYKMSMMYKRWRKWYWKYWMWRMRKRMYKYWWRRKRMMYMWYKKWMWKKRRRRWRMMSKRRYKRWMWYWKSWYKRKWRSYKMRRKRKYWWWKMYWSRMSKMWSSYWWSYSMYSYSSRMYSRSRMSSMWMYSYWKYSSMYRKSSWYRSYMMKRRRKRWYSMS"
all_match = True
for row in combined:
    ds = row["dataset"]
    seq = row["combined_consensus"]
    if seq == ref_seq:
        print(f"{ds}: True")
    else:
        all_match = False
        exp_len, got_len = len(ref_seq), len(seq)
        min_len = min(exp_len, got_len)
        diff_pos = None
        for i in range(min_len):
            if ref_seq[i] != seq[i]:
                diff_pos = i
                break
        if diff_pos is None and exp_len != got_len:
            diff_pos = min_len
        print(f"{ds}: MISMATCH (expected len {exp_len}, got {got_len}, first_diff={diff_pos})")

if not all_match:
    raise AssertionError("combined_consensus does not match ref_seq for all datasets")